# MoE + MLA Sizing (Practice Set)


In [2]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    n_embd: int
    n_head: int
    n_kv_heads: int
    n_layer: int
    rotary_ratio: float
    latent_kv_dim: int
    latent_q_dim: int
    num_experts: int
    num_shared_experts: int
    k: int                # routed experts activated per token
    vocab_size: int
    max_seq_len: int
    bias: bool

toy = Config(
    n_embd=64, n_head=4, n_kv_heads=2, n_layer=4, rotary_ratio=0.25,
    latent_kv_dim=32, latent_q_dim=32,
    num_experts=4, num_shared_experts=2, k=2,
    vocab_size=65, max_seq_len=4096,   # vocab_size=65 for shakespeare_char
    bias=False,
)

# hypothetical, ~OLMoE scale (7B/1.4B)
ref = Config(
    n_embd=1024, n_head=16, n_kv_heads=4, n_layer=24, rotary_ratio=0.25,
    latent_kv_dim=256, latent_q_dim=768,
    num_experts=32, num_shared_experts=2, k=4,
    vocab_size=50000, max_seq_len=4096,
    bias=False
)

print(toy)
print(ref)

Config(n_embd=64, n_head=4, n_kv_heads=2, n_layer=4, rotary_ratio=0.25, latent_kv_dim=32, latent_q_dim=32, num_experts=4, num_shared_experts=2, k=2, vocab_size=65, max_seq_len=4096, bias=False)
Config(n_embd=1024, n_head=16, n_kv_heads=4, n_layer=24, rotary_ratio=0.25, latent_kv_dim=256, latent_q_dim=768, num_experts=32, num_shared_experts=2, k=4, vocab_size=50000, max_seq_len=4096, bias=False)


### Part 1 — Params: total vs active
Q1.1) Calculate params per expert.

In [18]:
def param_count_per_expert(n_embd, mult=4, bias=True):
    expert_hidden = n_embd * mult
    p = 0
    # up-projection  Linear(n_embd, hidden)
    p += (n_embd * expert_hidden) + (expert_hidden if bias else 0)
    # down-projection Linear(hidden, n_embd)
    p += (expert_hidden * n_embd) + (n_embd if bias else 0)
    return p

print(f"P_expert(ref) = {param_count_per_expert(ref.n_embd, bias=ref.bias):,}")

P_expert(ref) = 8,388,608


Note: for swiglu style experts, you'll have to account for the gate layer also.

### Q1.2 — Router, then total vs active FFN params for one block

The router is `nn.Linear(n_embd, num_experts)` (bias passed from config).

For **one transformer block**, write:
- `ffn_total(cfg)` — every parameter that *exists* in the block's FFN: all `num_experts` routed experts + all `num_shared_experts` shared experts + the router. (Shared experts reuse the same `Expert` class — same formula, just always-on instead of gated.)
- `ffn_active(cfg)` — every parameter that actually *does a matmul* for a given token: `k` routed experts + all shared experts (they run on every token) + the router (it evaluates every token too — routing decisions aren't free).

Evaluate both for `ref`, print the ratio `ffn_active/ffn_total`.

In [20]:
def P_router(n_embd, num_experts):
    p = n_embd * num_experts
    return p

def ffn_total(cfg):
    pe = param_count_per_expert(cfg.n_embd, bias=cfg.bias)
    pr = P_router(cfg.n_embd, cfg.num_experts)
    p = pr + (cfg.num_experts + cfg.num_shared_experts) * pe
    return p

def ffn_active(cfg):
    pe = param_count_per_expert(cfg.n_embd, bias=cfg.bias)
    pr = P_router(cfg.n_embd, cfg.num_experts)
    return pr + (cfg.num_shared_experts + cfg.k) * pe

t, a = ffn_total(ref), ffn_active(ref)
print(f"ffn_total/block  = {t:,}")
print(f"ffn_active/block = {a:,}")
print(f"sparsity ratio    = {a/t:.3f}")

ffn_total/block  = 285,245,440
ffn_active/block = 50,364,416
sparsity ratio    = 0.177


### Q1.3 — design question

If we added a `expert_hidden` config knob (decoupled from `n_embd`) to this repo's `Expert`:
1. How does `param_count_per_expert` change? it gets smaller obv
2. For `ref`'s `num_experts=32`, what `expert_hidden` would keep `P_total` roughly where it is now (~7B) if we doubled `num_experts` to 64?

Roughly half of original hidden dim (which was 4*n_embd)

### Q1.4 — "active params" is a per-token idea, not a per-step idea

Suppose you train `ref` for one step with `batch_size=32, block_size=4096` (i.e. ~131K tokens in the batch). After `loss.backward()`, you inspect `model.blocks[0].ffn.experts[i].weight.grad for i in range(32)`.

Will roughly `k/num_experts` of them be `None` (never touched), or will almost **all 32** have a non-`None` gradient? Why doesn't the answer contradict everything you just computed in Q1.2?

My reasoning:  
* There are 32C4 ways for each token to select 4 experts out of 32.

* If a particular expert is excluded, the number of possible combinations is 31C4.

* Therefore, the probability that a token does not select a given expert is 31C4/32C4 = 0.875.

* Now the probability that none of the 131K tokens selects that expert is 0.875^131072, which is practically zero.

* Therefore, almost all 32 experts will have non-`None` gradients after the training step.


### Q1.5 — Attention params: `MLADeepSeekOptimized`




In [21]:
from collections import OrderedDict

In [22]:
def P_mla(cfg):
    dh = cfg.n_embd // cfg.n_head # head dimension
    dh_rotary = int(dh * cfg.rotary_ratio)
    dh_nr = dh - dh_rotary

    out = OrderedDict()
    out['mla/down_proj_kv'] = cfg.n_embd * cfg.latent_kv_dim
    out['mla/down_proj_q'] = cfg.n_embd * cfg.latent_q_dim
    out['mla/k_rotary'] = cfg.n_embd * dh_rotary
    out['mla/q_rotary'] = cfg.latent_q_dim * cfg.n_head*dh_rotary
    out['mla/up_k'] = cfg.latent_kv_dim * cfg.n_head*dh_nr
    out['mla/up_v'] = cfg.latent_kv_dim * cfg.n_head*dh
    out['mla/up_q'] = cfg.latent_q_dim * cfg.n_head*dh_nr
    out['mla/out_proj'] = cfg.n_embd * cfg.n_embd

    out['mla'] = out['mla/down_proj_kv'] + out['mla/down_proj_q'] + out['mla/k_rotary'] + out['mla/q_rotary'] + out['mla/up_k'] + out['mla/up_v'] + out['mla/up_q'] + out['mla/out_proj']
    return out['mla']

print(f"P_mla(ref)/block = {P_mla(ref):,}")


P_mla(ref)/block = 3,358,720


### Q1.6 — The whole model together

`model/gpt_moe.py`: token embedding (`vocab_size x n_embd`), `n_layer` blocks, then a **separate** (not weight-tied) `lm_head` of the same shape. (Assuming LayerNorm params as negligible).


In [23]:
def P_total(cfg):
    embed = cfg.vocab_size * cfg.n_embd
    lm_head = cfg.vocab_size * cfg.n_embd
    per_block = P_mla(cfg) + ffn_total(cfg)

    return embed + (cfg.n_layer * per_block) + lm_head

def P_active(cfg):
    embed = cfg.vocab_size * cfg.n_embd
    lm_head = cfg.vocab_size * cfg.n_embd
    per_block = P_mla(cfg) + ffn_active(cfg)
    return embed + (cfg.n_layer * per_block) + lm_head

pt, pa = P_total(ref), P_active(ref)
print(f"P_total(ref)  = {pt:,}  ({pt/1e9:.3f}B)")
print(f"P_active(ref) = {pa:,}  ({pa/1e9:.3f}B)")
print(f"ratio active/total = {pa/pt:.3f}")

P_total(ref)  = 7,028,899,840  (7.029B)
P_active(ref) = 1,391,755,264  (1.392B)
ratio active/total = 0.198


## Part 2 — FLOPs / token

### Q2.1 — Which N?

Each parameter in a matmul does one multiply and one add per token, so the forward pass costs 2 FLOPs per parameter, giving 2N. The backward pass needs two matmuls of the same size, one for gradients w.r.t. activations and one for gradients w.r.t. weights, so it costs 2 × 2N = 4N, and fwd + bwd = 2N + 4N = 6N.

For MoE models, N should be the active non-embedding params, since only those do matmuls per token. Attention's score computation (the T×T part) has no parameters, so 6N ignores it, which is fine at short context but understates cost at long context.


In [ ]:
flops_train_active = 6 * P_active(ref)
print(f"6*N_active = {flops_train_active/1e9:.3f} GFLOPs/token")

6*N_active = 8.355 GFLOPs/token


### Q2.2 — Is the router's own compute worth worrying about?

The router itself is a real matmul: Linear(n_embd, num_experts), run on every token, every layer. Its forward FLOPs/token (all layers) = 2 * n_embd * num_experts * n_layer.


In [ ]:
router_flops = 2 * ref.n_embd * ref.num_experts * ref.n_layer
fwd_active_flops = 2 * P_active(ref)
print(f"router FLOPs/token = {router_flops:,}")
print(f"as % of forward active FLOPs: {router_flops/fwd_active_flops*100:.4f}%")

router FLOPs/token = 1,572,864
as % of forward active FLOPs: 0.0565%


note: it's tiny, so we can ignore routing overhead in the back-of-the-envelope compute budget

### Q2.3 — The whole point of MoE


| model | N used | FLOPs/token |
|---|---|---|
| (a) dense, sized to `1.4B` | `1.4B` | ? |
| (b) dense, sized to `7B` | `7B` | ? |
| (c) `ref` (the actual MoE) | `1.4B` | ? |
___

What does an MoE model buy you that a dense model of equal FLOPs-cost doesn't? **Capacity**.  
Our MoE stores the knowledge of a 7B model but pays the per-token compute of a 1.4B one, so you get more capacity per FLOP. The router doesn't make the model "smarter" by itself; it just picks which 1.4B slice of the 7B to use for each token, so different tokens can draw on different specialized parameters.


In [ ]:
flops_a = 2 * P_active(ref)
flops_b = 2 * P_total(ref)
flops_c = 2 * P_active(ref)

print(f"(a) dense @ P_active : {flops_a/1e9:.3f} GFLOPs/token")
print(f"(b) dense @ P_total  : {flops_b/1e9:.3f} GFLOPs/token")
print(f"(c) ref MoE          : {flops_c/1e9:.3f} GFLOPs/token")
print(f"(b)/(a) = {flops_b/flops_a:.2f}x more compute for a dense model to match ref\\'s TOTAL capacity")

(a) dense @ P_active : 2.785 GFLOPs/token
(b) dense @ P_total  : 14.066 GFLOPs/token
(c) ref MoE          : 2.785 GFLOPs/token
(b)/(a) = 5.05x more compute for a dense model to match ref\'s TOTAL capacity


## Part 3 — Attention memory: the MLA payoff (KV cache)


In [4]:
def kv_cache_elems_per_token_per_layer(cfg, n_kv_heads):
    dh = cfg.n_embd // cfg.n_head # head dimension
    return 2 * dh * n_kv_heads

mha_elems = kv_cache_elems_per_token_per_layer(ref, n_kv_heads=ref.n_head)
gqa_elems = kv_cache_elems_per_token_per_layer(ref, n_kv_heads=ref.n_kv_heads)
print(f"MHA elems/token/layer = {mha_elems}")
print(f"GQA elems/token/layer = {gqa_elems}")

MHA elems/token/layer = 2048
GQA elems/token/layer = 512


In [6]:
def mla_cache_elems_per_token_per_layer(cfg):
    dh = cfg.n_embd // cfg.n_head
    dh_rotary = int(dh * cfg.rotary_ratio)
    return cfg.latent_kv_dim + dh_rotary

print(f"MLA elems/token/layer = {mla_cache_elems_per_token_per_layer(ref)}")

MLA elems/token/layer = 272


### Q. Total KV-cache footprint (GB) for the whole model


In [7]:
BYTES_PER_ELEM = 2  # bf16

def total_kv_cache_bytes_per_token(elems_per_token_per_layer, cfg):
    total = elems_per_token_per_layer * cfg.n_layer * BYTES_PER_ELEM
    return total

mla_bpt = total_kv_cache_bytes_per_token(mla_cache_elems_per_token_per_layer(ref), ref)
gqa_bpt = total_kv_cache_bytes_per_token(gqa_elems, ref)
mha_bpt = total_kv_cache_bytes_per_token(mha_elems, ref)

print(f"bytes/token (all layers): MHA={mha_bpt:,}  GQA={gqa_bpt:,}  MLA={mla_bpt:,}")
print(f"MLA compression vs MHA: {mha_bpt/mla_bpt:.2f}x   vs GQA: {gqa_bpt/mla_bpt:.2f}x")

batch_size, seqlen = 32, ref.max_seq_len
for name, bpt in [("MHA", mha_bpt), ("GQA", gqa_bpt), ("MLA", mla_bpt)]:
    gb = (bpt * seqlen * batch_size)/1e9
    print(f"  {name}: {gb:.2f} GB for {batch_size} sequences @ {seqlen} tokens")

bytes/token (all layers): MHA=98,304  GQA=24,576  MLA=13,056
MLA compression vs MHA: 7.53x   vs GQA: 1.88x
  MHA: 12.88 GB for 32 sequences @ 4096 tokens
  GQA: 3.22 GB for 32 sequences @ 4096 tokens
  MLA: 1.71 GB for 32 sequences @ 4096 tokens


### Q. Does `use_weight_absorption=True` change the FLOP count?


- **Without** absorption: `up_k`/`up_v` reconstruct full `K`/`V` (shape `..., n_head, dh`) every call, then a normal `Q @ K^T`.
- **With** absorption: `Wq = up_q.weight.view(...)`, `Wk = up_k.weight.view(...)`, and the attention score is computed as `einsum("btq,hdq,hdk->bhtk", cq, Wq, Wk)` directly in **latent space** (dimension `latent_kv_dim`, not `dh`), and similarly `up_v` gets folded into `out_proj` on the output side.


1. Is weight absorption primarily a **FLOPs** optimization, a **memory-bandwidth / cache-size** optimization, or both?
2. During incremental decoding (`T=1`, growing `L`), which quantity do you avoid materializing every step by using absorption, and why does that matter more as `L` grows?


## Part 4 — Training memory: weights, optimizer, activations

### Q4.1 — Weights + optimizer state

- Weight memory (bf16) = `2 bytes * P_total` — **not** `P_active`. Every expert's weights live on the GPU whether or not they fired this step.
- AdamW optimizer state ≈ `12 bytes/param` on top of that (fp32 master weight + first + second moment, 4 bytes each) — and per Q1.4, this is also over `P_total`, because across a real batch essentially every expert receives *some* gradient.

Compute total weights+optimizer GB for `ref`. This number is why **MoE training is memory-hungry. It's FLOPs-cheap, you're paying full price in HBM for capacity you're only paying a fraction of in compute.**

Note: BF16 weights (2 bytes): Reduce memory usage and enable faster matrix multiplications on GPUs with specialized BF16 hardware, such as NVIDIA A100s and H100s. BF16 also has a wide exponent range, helping avoid numerical overflow.

FP32 master weights (4 bytes): Preserve small weight updates that might otherwise be rounded away in BF16. The optimizer updates the FP32 copy, then the updated weights are cast to BF16 for computation.


In [ ]:
weight_bytes = 2 * P_total(ref)
optimizer_bytes = 12 * P_total(ref)

print(f"weights (bf16):          {weight_bytes/1e9:.2f} GB")
print(f"optimizer state (AdamW): {optimizer_bytes/1e9:.2f} GB")
print(f"total:                   {(weight_bytes+optimizer_bytes)/1e9:.2f} GB")

weights (bf16):          14.07 GB
optimizer state (AdamW): 84.40 GB
total:                   98.46 GB


wow!  

note: This already doesn't fit on one 80 GB GPU, and it still excludes gradients (+2 bytes/param in bf16, or +4 in fp32, so 16-18 bytes/param in total) and activations. You'd need sharding (FSDP/ZeRO) or expert parallelism, which is why the memory bill scales with P_total while the compute bill scales with P_active.

### Q4.2 — Activation memory

Look at how your moe's `forward()` actually computes the routed path:

```python
token_idx, slot_idx = (topk_idx == expert_id).nonzero(as_tuple=True)
expert_input = tokens[token_idx]        # gather only this expert's tokens
expert_output = expert(expert_input)    # forward only on those
```

Because it **gathers** exactly the tokens routed to each expert, activation memory for the routed path scales with *active* compute, not total — you never materialize an activation tensor for a token going through an expert it wasn't routed to.

Production MoE kernels (Megatron, DeepSpeed-MoE, etc.) usually do something different for GPU efficiency: give every expert a **fixed capacity buffer** (sized for the *expected* load, e.g. `capacity_factor * tokens_per_batch / num_experts`), pad underfilled experts, and **drop** tokens that overflow a full expert — trading a small amount of quality/tokens for predictable, batchable tensor shapes.

Question: what tradeoff is your exact-gather approach making relative to a capacity-buffer approach — in terms of memory predictability, GPU utilization, and correctness? And how does `MoEDeepSeek.update_expert_bias()` (the aux-loss-free routing bias) reduce how often you'd actually hit that overflow case?


## Part 5 — MFU

- MFU = (achieved_FLOP/s) / (peak_hardware_FLOP/s).
- achieved_FLOP/s = (FLOPs/token) * (tokens/sec)

For an MoE model, if you plugged in 6P_total instead of 6*P_active for "FLOPs/token", would your reported MFU go up or down — and is that number now measuring something real, or something misleading? MFU would go up, but it would be misleading because using 6P
total overestimates the actual FLOPs executed per token, making the reported MFU artificially inflated:(


For MoE, a high MFU is useful, but memory bandwidth, GPU utilization, and inter-GPU communication are also critical bottlenecks, so don't spend too much time worrying about mfu

Q. A GPU pod reports 450 ms/iteration training ref-shaped model at batch_size=8, block_size=4096, on a single H100 (bf16 peak ≈ 990 TFLOP/s, dense — i.e. not counting any sparsity credit).

Compute: tokens/sec, achieved FLOP/s and MFU%.

In [29]:
B, T, ms_iter = 8, 4096, 450
peak_flops = 990e12
P_non_embedding = P_active(ref) - (ref.vocab_size * ref.n_embd) # embedding lookup isnt a matmul

tokens_per_sec = (B * T * 1000)/ms_iter
achieved_flops = 6 * P_non_embedding * tokens_per_sec
mfu = achieved_flops / peak_flops

print(f"tokens/sec:     {tokens_per_sec:,.0f}")
print(f"achieved FLOP/s: {achieved_flops:.3e}")
print(f"MFU:            {mfu*100:.2f}%")

tokens/sec:     72,818
achieved FLOP/s: 5.857e+14
MFU:            59.16%


note: The attention T×T term is ignored. At T=4096 it adds roughly 12 · n_layer · n_embd · T ≈ 1.2e9 FLOPs/token (about 14% on top of 6N), so PaLM-style accounting would push MFU higher